# Persistence and Streaming

In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
# pip install langchain-community

In [3]:
# pip install langchain-tavily

In [4]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

In [5]:
tool = TavilySearch(max_results=2)

In [6]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [7]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [8]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [9]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [10]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [11]:
thread = {"configurable": {"thread_id": "1"}}

In [12]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 1259, 'total_tokens': 1283, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2fe17714a7', 'id': 'chatcmpl-EAVqsJekWzh3S4q2ieEBEJu2UUOfG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe04a-c861-7e82-b046-b361814005a3-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'San Francisco weather today', 'search_depth': 'fast'}, 'id': 'call_lGHzIsU2evh26GTOQirVpbjf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1259, 'output_tokens': 24, 'total_tokens': 1283, 'input_token_details': {'audio': 0, 'cache_

In [13]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 2073, 'total_tokens': 2097, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1920}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2fe17714a7', 'id': 'chatcmpl-EAVqwyZYUgOl50IdrZ7HB6CUZv73c', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe04a-dbd4-7cb1-ae58-d5bf8c52705f-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Los Angeles weather today', 'search_depth': 'fast'}, 'id': 'call_iTY6FEMFH8ds94KiPCtzXw6M', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2073, 'output_tokens': 24, 'total_tokens': 2097, 'input_token_details': {'audi

In [14]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Based on the typical temperature patterns and the data from the search, Los Angeles tends to be warmer than San Francisco. For specific temperatures today, you would need to check local weather services for the most accurate and current data on temperatures in both cities.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 3006, 'total_tokens': 3056, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 2944}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2fe17714a7', 'id': 'chatcmpl-EAVr0ESECmmK7RVRb1Q47nTAhdFHM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe04a-e96d-7341-9f30-e191bf72c632-0', tool_calls=[], invalid_tool_calls=[], 

In [15]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Could you please clarify what you are comparing to determine which one is warmer? Are you referring to two specific locations, materials, clothing items, or something else?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 1257, 'total_tokens': 1290, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2fe17714a7', 'id': 'chatcmpl-EAVr1yhJvcWmDgCVLJpNlXocDJjUQ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe04a-efba-7aa2-8473-5b1d8e398e5a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1257, 'output_tokens': 33, 'total_tokens': 1290, 'input_token_details': 

## Streaming tokens

In [16]:
# MemorySaver supports both sync and async — reuse the same instance
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [17]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

Calling: {'name': 'tavily_search', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_RRCJDUuSRNzC4tE9AAMJM4ub', 'type': 'tool_call'}
Back to the model!
The| current| weather| in| San| Francisco| is| over|cast| with| a| temperature| of| |13|.|2|°C| (|55|.|8|°F|).| The| wind| is| coming| from| the| west|-s|outh|west| at| |6|.|9| mph| (|11|.|2| k|ph|).| Hum|idity| is| at| |100|%,| and| visibility| is| |10| kilometers|.| There| is| no| precipitation| at| the| moment|.|